# ðŸŽ“ NB3 â€” Inference & Evaluation (v3)

**Prerequisites**: NB1 + NB2 must have completed. Shared `src/` + `config/` must be available.
**Systems**: (a) zero-shot Qwen3-4B  (b) zero-shot Qwen3-8B  (c) fine-tuned Qwen3-4B
**Run order**: NB1 â†’ NB2 â†’ NB3

## Section 0 â€” Environment Setup (no restart required)

In [ ]:
# ============================================================================
# 0A. INSTALL
# ============================================================================
import subprocess, sys, importlib

print("Installing / upgrading â€¦")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "huggingface_hub", "transformers", "peft", "accelerate", "bitsandbytes>=0.45.0", "scikit-learn",
], stdout=subprocess.DEVNULL)

# IMPORTANT: do NOT purge bitsandbytes â€” it registers PyTorch custom operators at
# import time; evicting it from sys.modules causes a fatal duplicate-registration
# error when any subsequent code re-imports it.
for _k in list(sys.modules.keys()):
    if any(_k == _pkg or _k.startswith(_pkg + ".") for _pkg in ["huggingface_hub", "transformers", "peft", "accelerate"]):
        sys.modules.pop(_k, None)
importlib.invalidate_caches()

# Hugging Face hub compatibility shim (ensures is_offline_mode is accessible)
try:
    import huggingface_hub
    if not hasattr(huggingface_hub, "is_offline_mode"):
        try:
            from huggingface_hub.utils import is_offline_mode
            huggingface_hub.is_offline_mode = is_offline_mode
        except Exception:
            import os
            huggingface_hub.is_offline_mode = lambda: os.getenv("HF_HUB_OFFLINE") == "1"
except Exception:
    pass

import importlib as _il
for _pkg in ["huggingface_hub", "bitsandbytes", "peft", "transformers", "sklearn"]:
    try:
        _m = _il.import_module(_pkg)
        print(f"  {_pkg:20s}  {getattr(_m,'__version__','?')}")
    except Exception as _e:
        print(f"  {_pkg:20s}  ERROR: {_e}")

In [ ]:
# ============================================================================
# 0B. PREFLIGHT
# ============================================================================
import os, torch

failures = []

if not torch.cuda.is_available():
    failures.append("CUDA not visible.")
else:
    n_gpu = torch.cuda.device_count()
    print(f"[PASS] {n_gpu} GPU(s): " + ", ".join(torch.cuda.get_device_name(i) for i in range(n_gpu)))

try:
    import bitsandbytes as _bnb
    from packaging.version import Version
    if Version(_bnb.__version__) < Version("0.45.0"):
        failures.append(f"bitsandbytes {_bnb.__version__} < 0.45.0")
    else:
        print(f"[PASS] bitsandbytes {_bnb.__version__}")
except RuntimeError as _e:
    if "register" in str(_e) or "Duplicate" in str(_e):
        _bnb_cached = sys.modules.get("bitsandbytes")
        _bnb_ver = getattr(_bnb_cached, "__version__", None) if _bnb_cached else None
        if _bnb_ver:
            print(f"[PASS] bitsandbytes {_bnb_ver} (operators pre-registered, using cached module)")
        else:
            print("[WARN] bitsandbytes: duplicate operator registration (harmless â€” operators already loaded)")
    else:
        failures.append(f"bitsandbytes: {_e}")
except Exception as _e:
    failures.append(f"bitsandbytes: {_e}")

if failures:
    raise RuntimeError("PREFLIGHT FAILED:\n" + "\n".join(f"  â€¢ {f}" for f in failures))
print("\nAll preflight checks passed.\n")

In [ ]:
# ============================================================================
# 0C. ALL IMPORTS
# ============================================================================
import os, sys, json, re, random, logging, csv, warnings, gc, time
import dataclasses as _dc
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict, Counter

import numpy as np
from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import cohen_kappa_score, mean_absolute_error

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("nb3_infer")

# Silence noisy HTTP and telemetry loggers
for _noisy_logger in ["httpx", "huggingface_hub", "urllib3", "urllib3.connectionpool"]:
    logging.getLogger(_noisy_logger).setLevel(logging.WARNING)


In [ ]:
# ============================================================================
# 0C-bis. SHARED CODE (src/ — one copy of schema, prompt and paths; README §5)
# ============================================================================
import os, sys, glob as _glob, shutil, importlib, builtins

# Clear ALL stale cached src.* modules from previous cell attempts
for _m in list(sys.modules.keys()):
    if _m == "src" or _m.startswith("src."):
        sys.modules.pop(_m, None)
importlib.invalidate_caches()

# ---- Canonical schema.py (standalone v3.1) ----
_CANONICAL_SCHEMA = r'''"""Single source of truth for the grading output schema and its validator.

Every notebook (NB1 data, NB2 training, NB3 evaluation) imports this module.
Do not copy the validator into a notebook; import it.
"""
import json
import os
import re
from typing import List, Optional, Tuple

_SCHEMA_FILE = os.path.join(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
    "config", "schema_version.txt",
)


def _read_schema_version() -> str:
    if not os.path.isfile(_SCHEMA_FILE):
        return "v3.1"
    try:
        with open(_SCHEMA_FILE, "r", encoding="utf-8") as f:
            v = f.read().strip()
            return v if v else "v3.1"
    except Exception:
        return "v3.1"


SCHEMA_VERSION: str = _read_schema_version()

# A deduction with no evidence lines must explain what is absent (README 6.5).
ABSENCE_MARKERS: Tuple[str, ...] = (
    "missing", "omit", "absent", "lacks", "not found", "fails to",
    "unhandled", "unreadable", "without", "no ", "never",
)

# Numbered student code uses "<n>| <code>" (see prompts.add_line_numbers).
_LINE_PREFIX_RE = re.compile(r"^\s*\d+\|\s?")


def strip_line_number(numbered_line: str) -> str:
    """Remove the '<n>| ' prefix from one numbered line."""
    return _LINE_PREFIX_RE.sub("", numbered_line, count=1)


def has_absence_marker(comment: str) -> bool:
    low = comment.lower()
    return any(m in low for m in ABSENCE_MARKERS)


def validate_grading_output(output_json: str, rubric: list, student_answer_lineno: str,
                            execution_report: Optional[dict] = None) -> Tuple[bool, List[str]]:
    """Validate one grading output against the rubric and the numbered student code.

    Returns (ok, errors). Checks: JSON object with exactly the schema keys,
    no 'total'/'max', known + unique + complete criterion ids, marks in
    [0, max] and 0.5 steps, a substantive comment, integer evidence lines that
    exist and are not blank, an absence explanation for evidence-free
    deductions, and (if an execution report is given) no full compile marks
    when compilation failed.
    """
    errors: List[str] = []
    try:
        data = json.loads(output_json)
    except (json.JSONDecodeError, TypeError) as e:
        return False, [f"JSON parse error: {e}"]

    if not isinstance(data, dict):
        return False, ["Output is not a JSON object"]

    for key in ("criteria", "needs_human_review"):
        if key not in data:
            errors.append(f"Missing key '{key}'")
    if errors:
        return False, errors

    if not isinstance(data["criteria"], list):
        return False, ["'criteria' must be a list"]
    if not isinstance(data["needs_human_review"], bool):
        errors.append("'needs_human_review' must be a boolean")
    if "total" in data:
        errors.append("Output must not contain 'total' (sum in code)")
    if any("max" in c for c in data["criteria"] if isinstance(c, dict)):
        errors.append("Criteria must not contain 'max' (comes from rubric)")

    rubric_map = {c["id"]: c for c in rubric}
    rubric_ids = set(rubric_map)
    output_ids = set()

    answer_lines = student_answer_lineno.split("\n")
    n_lines = len(answer_lines)

    for crit in data["criteria"]:
        if not isinstance(crit, dict):
            errors.append("Criterion is not a dict")
            continue
        for k in ("id", "awarded", "comment", "evidence_lines"):
            if k not in crit:
                errors.append(f"Criterion missing key '{k}'")

        cid = crit.get("id")
        if cid not in rubric_ids:
            errors.append(f"Unknown criterion id '{cid}'")
            continue
        if cid in output_ids:
            errors.append(f"Duplicate criterion id '{cid}'")
            continue
        output_ids.add(cid)

        awarded = crit.get("awarded", -1)
        max_m = rubric_map[cid]["max_marks"]
        awarded_is_num = isinstance(awarded, (int, float)) and not isinstance(awarded, bool)
        if not awarded_is_num:
            errors.append(f"'{cid}': awarded not a number")
        else:
            if awarded < 0 or awarded > max_m:
                errors.append(f"'{cid}': awarded {awarded} not in [0, {max_m}]")
            if (awarded * 2) % 1 != 0:
                errors.append(f"'{cid}': awarded {awarded} not in 0.5 increments")

        comment = crit.get("comment", "")
        if not isinstance(comment, str) or not comment.strip():
            errors.append(f"'{cid}': comment must be non-empty")
            comment = ""
        elif len(comment.split()) < 3:
            errors.append(f"'{cid}': comment must be a substantive explanation (>= 3 words)")

        ev_lines = crit.get("evidence_lines", [])
        if not isinstance(ev_lines, list):
            errors.append(f"'{cid}': evidence_lines must be a list")
            continue
        for ln in ev_lines:
            if not isinstance(ln, int) or isinstance(ln, bool):
                errors.append(f"'{cid}': evidence_lines must contain integers, got {ln!r}")
            elif ln < 1 or ln > n_lines:
                errors.append(f"'{cid}': line {ln} out of range [1, {n_lines}]")
            elif not strip_line_number(answer_lines[ln - 1]).strip():
                errors.append(f"'{cid}': cited evidence line {ln} is empty/blank (ungrounded citation)")

        if awarded_is_num and awarded < max_m and not ev_lines and comment:
            if not has_absence_marker(comment):
                errors.append(
                    f"'{cid}': deduction made without evidence_lines, but comment "
                    "does not explain what is missing/absent"
                )

    missing = rubric_ids - output_ids
    if missing:
        errors.append(f"Missing criteria: {sorted(missing)}")

    if isinstance(execution_report, dict) and execution_report.get("compile_ok") is False:
        for c in data["criteria"]:
            if not isinstance(c, dict):
                continue
            cid = str(c.get("id", ""))
            if any(t in cid.lower() for t in ("comp", "syntax", "compile")):
                if c.get("awarded") == rubric_map.get(cid, {}).get("max_marks"):
                    errors.append(f"'{cid}': full marks awarded despite compile failure in execution report")

    return len(errors) == 0, errors
'''

# ---- Canonical prompts.py (standalone v3.1) ----
_CANONICAL_PROMPTS = r'''"""Single grading prompt builder (README sections 5, 6.5, 6.6).

The exact same text must be used to build training data (NB1), train (NB2)
and evaluate / infer (NB3). Import from here; never copy.
"""
import json
from typing import Optional

from .schema import SCHEMA_VERSION

SYSTEM_PROMPT = """\
You are an expert C++ Programming grader (schema {schema_version}).
Grade the student's answer against the rubric below.

RULES
1. Evaluate each criterion independently.
2. Award marks only for what the student's code actually demonstrates.
3. comment: ≤20 words explaining the assessment.
4. evidence_lines: list of 1-based line numbers in the student code where the
   issue appears, or [] when the criterion is fully met or the issue is
   MISSING code (explain in comment what is absent).
5. Output ONLY compact JSON matching the schema — no markdown, no extra text.

OUTPUT SCHEMA (schema {schema_version})
{{"criteria":[{{"id":"<id>","awarded":<number>,"comment":"<string>","evidence_lines":[<int>,...]}},...],"needs_human_review":false}}

NOTES
- awarded must be in 0.5 increments and in [0, criterion_max].
- total is NOT in the output; it is computed from awarded values.
- max is NOT in the output; it comes from the rubric.
""".format(schema_version=SCHEMA_VERSION)


def add_line_numbers(code: str) -> str:
    """Number lines as '<n>| <code>', width = digits of the line count."""
    lines = code.split("\n")
    width = len(str(len(lines)))
    return "\n".join(f"{i + 1:{width}}| {line}" for i, line in enumerate(lines))


def build_exec_section(rep: Optional[dict]) -> str:
    """Render the execution report. A missing report is never shown as a compile failure."""
    if not isinstance(rep, dict) or not rep or rep.get("available") is False:
        return '{"available": false}'
    if not rep.get("compile_ok"):
        return json.dumps({
            "compile_ok": False,
            "compile_error": (rep.get("compile_error") or "unknown")[:400],
            "compile_lines": rep.get("compile_lines", [])[:10],
        })
    results = []
    for tc in rep.get("test_results", [])[:6]:
        results.append({
            "stdin": (tc.get("stdin") or "")[:120],
            "expected": (tc.get("expected") or "")[:120],
            "actual": (tc.get("actual") or "")[:120],
            "passed": tc.get("passed", False),
            "exit_code": tc.get("exit_code"),
            "timeout": tc.get("timeout", False),
        })
    return json.dumps({
        "compile_ok": True,
        "tests_passed": rep.get("tests_passed", 0),
        "tests_total": rep.get("tests_total", 0),
        "runtime_errors": [str(e)[:120] for e in rep.get("runtime_errors", [])[:4]],
        "test_results": results,
    })


def build_grading_messages(question_text: str, rubric: list, student_answer_lineno: str,
                           execution_report: Optional[dict]) -> list:
    """System + user chat messages for one grading request."""
    rub = json.dumps(
        [{"id": c["id"], "description": c["description"], "max_marks": c["max_marks"]}
         for c in rubric],
        indent=2,
    )
    user_msg = (
        f"## Question\n{question_text}\n\n"
        f"## Rubric\n{rub}\n\n"
        f"## Student Answer (with line numbers)\n```\n{student_answer_lineno}\n```\n\n"
        f"## Execution Report\n{build_exec_section(execution_report)}\n\n"
        "Grade this answer. Output ONLY the compact JSON."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]


def build_gold_output(gold_criteria: list, needs_human_review: bool) -> str:
    """Compact gold JSON target (no total, no max)."""
    return json.dumps(
        {"criteria": gold_criteria, "needs_human_review": needs_human_review},
        ensure_ascii=False, separators=(",", ":"),
    )
'''

# ---- Canonical paths.py ----
_CANONICAL_PATHS = r'''"""Where Phase 1 writes its dataset and where later phases read it.

NB1 writes to <BASE_DIR>/data_v1_1. NB2, NB3 and the Phase 0 audit call
resolve_data_dir(), which never guesses silently: it uses DATA_DIR if set,
otherwise the NB1 output folder, otherwise exactly one attached Kaggle
dataset that contains data_manifest.json. Anything else raises.
"""
import glob
import os

DATASET_DIRNAME = "data_v1_1"
MANIFEST_NAME = "data_manifest.json"


def base_dir() -> str:
    env = os.environ.get("BASE_DIR")
    if env:
        return env.rstrip("/\\")
    return "/kaggle/working" if os.path.isdir("/kaggle/working") else "."


def nb1_output_dir() -> str:
    """Folder NB1 writes the dataset into."""
    return os.path.join(base_dir(), DATASET_DIRNAME)


def resolve_data_dir(required_file: str = MANIFEST_NAME) -> str:
    """Folder later phases read the NB1 dataset from."""
    env = os.environ.get("DATA_DIR")
    if env:
        if not os.path.isfile(os.path.join(env, required_file)):
            raise FileNotFoundError(f"DATA_DIR={env} does not contain {required_file}.")
        return env

    local = nb1_output_dir()
    if os.path.isfile(os.path.join(local, required_file)):
        return local

    found = sorted({os.path.dirname(p) for p in
                    glob.glob(f"/kaggle/input/**/{required_file}", recursive=True)})
    if len(found) == 1:
        return found[0]
    if len(found) > 1:
        v1_1 = [p for p in found if "data_v1_1" in p]
        return v1_1[0] if v1_1 else found[0]
    raise FileNotFoundError(
        f"No NB1 dataset found (looked for {required_file} in {local} and /kaggle/input). "
        "Run NB1 (nb1_data.py) or attach its output dataset, or set DATA_DIR."
    )
'''

def _setup_project_root() -> str:
    pkg_dir = "/kaggle/working/cpp_grader_pkg" if os.path.isdir("/kaggle/working") else os.path.join(os.getcwd(), "cpp_grader_pkg")
    os.makedirs(os.path.join(pkg_dir, "src"), exist_ok=True)
    os.makedirs(os.path.join(pkg_dir, "config"), exist_ok=True)

    # 1. Prefer copying from attached dataset if available
    src_cands = [p for p in _glob.glob("/kaggle/input/**/src/prompts.py", recursive=True) if "working" not in p]
    if src_cands:
        src_parent = os.path.dirname(os.path.dirname(src_cands[0]))
        for item in ["src", "config"]:
            sp = os.path.join(src_parent, item)
            dp = os.path.join(pkg_dir, item)
            if os.path.isdir(sp):
                for f in os.listdir(sp):
                    shutil.copy2(os.path.join(sp, f), os.path.join(dp, f))
        print(f"[SETUP] Copied shared code from attached dataset: {src_parent}")
    else:
        # 2. Standalone fallback: write canonical code directly
        with open(os.path.join(pkg_dir, "src", "prompts.py"), "w", encoding="utf-8") as _pf:
            _pf.write(_CANONICAL_PROMPTS)
        with open(os.path.join(pkg_dir, "src", "paths.py"), "w", encoding="utf-8") as _ptf:
            _ptf.write(_CANONICAL_PATHS)
        with open(os.path.join(pkg_dir, "src", "schema.py"), "w", encoding="utf-8") as _sf:
            _sf.write(_CANONICAL_SCHEMA)
        with open(os.path.join(pkg_dir, "src", "__init__.py"), "w", encoding="utf-8") as _if:
            _if.write("")
        sv_dst = os.path.join(pkg_dir, "config", "schema_version.txt")
        with open(sv_dst, "w", encoding="utf-8") as _svf:
            _svf.write("v3.1\n")

    return pkg_dir

PROJECT_ROOT = _setup_project_root()

sys.path = [
    p for p in sys.path
    if not ("kaggle/input" in p and "cpp_grader_pkg" in p)
]
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Shared code: {PROJECT_ROOT}")

from src.schema import SCHEMA_VERSION, validate_grading_output
from src.prompts import (
    SYSTEM_PROMPT, add_line_numbers, build_exec_section,
    build_grading_messages, build_gold_output as _build_gold_json
)
from src.paths import resolve_data_dir

# Export to builtins for guaranteed cross-cell accessibility
builtins.validate_grading_output = validate_grading_output
builtins.SCHEMA_VERSION = SCHEMA_VERSION
builtins.SYSTEM_PROMPT = SYSTEM_PROMPT
builtins.add_line_numbers = add_line_numbers
builtins.build_exec_section = build_exec_section
builtins.build_grading_messages = build_grading_messages
builtins.resolve_data_dir = resolve_data_dir

print(f"[PASS] Shared code ready (SCHEMA_VERSION={SCHEMA_VERSION}).")


In [ ]:
# ============================================================================
# 0D. CONFIG (Auto-detect DATA_DIR & ADAPTER_DIR for full or smoke run)
# ============================================================================
import os, sys, glob as _glob, shutil as _shutil, json, random, re
import numpy as np
from dataclasses import dataclass, field

# ---------------------------------------------------------------------------
# Inline fallback for resolve_data_dir & SCHEMA_VERSION
# ---------------------------------------------------------------------------
if "SCHEMA_VERSION" not in globals():
    SCHEMA_VERSION = "v3.1"

# Shared code import
try:
    from src.paths import resolve_data_dir, nb1_output_dir
except Exception:
    pass

def resolve_data_dir(required_file="data_manifest.json"):
    env = os.environ.get("DATA_DIR")
    if env:
        if os.path.isfile(os.path.join(env, required_file)):
            return env
        sub = os.path.join(env, "data_v1_1")
        if os.path.isfile(os.path.join(sub, required_file)):
            return sub

    cands = sorted({os.path.dirname(p) for p in
                    _glob.glob(f"/kaggle/input/**/{required_file}", recursive=True)})
    if not cands:
        cands = sorted({os.path.dirname(p) for p in
                        _glob.glob(f"**/{required_file}", recursive=True)})

    # Prefer directories that actually contain the splits
    complete = [d for d in cands if all(os.path.isfile(os.path.join(d, f"{s}.jsonl"))
                                        for s in ["val", "test_seen", "test_unseen"])]
    if complete:
        v2_cand = [d for d in complete if "data-v2" in d or "data_v2" in d]
        if v2_cand: return v2_cand[0]
        v1_1 = [d for d in complete if "data_v1_1" in d or "data-v1-1" in d]
        return v1_1[0] if v1_1 else complete[0]
    if cands:
        v1_1 = [d for d in cands if "data_v1_1" in d]
        return v1_1[0] if v1_1 else cands[0]

    raise FileNotFoundError(
        f"No NB1 dataset found (looked for {required_file} in /kaggle/input). "
        "Attach the data dataset or set DATA_DIR."
    )

# ---------------------------------------------------------------------------
# 1. Config Object
# ---------------------------------------------------------------------------
@dataclass
class InferConfig:
    seed: int = 42
    student_model_id: str = "Qwen/Qwen3-4B"
    teacher_model_id: str = "Qwen/Qwen3-8B"
    batch_size:           int   = 8
    max_new_tokens:       int   = 512
    inference_retries:    int   = 2
    retry_temperature:    float = 0.3
    smoke_test:           bool  = False   # <<< FULL RUN: evaluates all samples across val, test_seen, test_unseen
    smoke_eval_samples:   int   = 30
    data_dir:   str = ""
    output_dir: str = "output"


CFG = InferConfig()
SMOKE_TEST = CFG.smoke_test

IS_KAGGLE = os.path.exists("/kaggle/working")
BASE_DIR  = "/kaggle/working" if IS_KAGGLE else "."
CFG.data_dir   = resolve_data_dir()
CFG.output_dir = os.path.join(BASE_DIR, CFG.output_dir)
os.makedirs(CFG.output_dir, exist_ok=True)
os.environ["DATA_DIR"] = CFG.data_dir

# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# 2. ADAPTER — Explicit config (Round 3: no auto-discovery, V4)
# ---------------------------------------------------------------------------
# Set EVAL_ARM and ADAPTER_PATH as Kaggle environment variables before running.
# Example: EVAL_ARM=A1, ADAPTER_PATH=/kaggle/input/my-adapter/best_adapter
CFG.eval_arm    = os.environ.get('EVAL_ARM', '')
CFG.adapter_path = os.environ.get('ADAPTER_PATH', '')
assert CFG.eval_arm in {'A0', 'A1', 'A2', 'A3'}, (
    f"Set EVAL_ARM environment variable to one of A0/A1/A2/A3; got: '{CFG.eval_arm}'"
)
assert CFG.adapter_path and os.path.isdir(CFG.adapter_path), (
    f"Set ADAPTER_PATH to a valid adapter directory; got: '{CFG.adapter_path}'"
)
BEST_ADAPTER = CFG.adapter_path
print(f'[ADAPTER] Arm={CFG.eval_arm} | Path={BEST_ADAPTER}')

random.seed(CFG.seed)
np.random.seed(CFG.seed)

print(f"SCHEMA_VERSION = {SCHEMA_VERSION}")
print(f"SMOKE_TEST     = {SMOKE_TEST}  (False = full evaluation on all samples)")
print(f"Data dir       = {CFG.data_dir}")

# ---- Verify adapter -------------------------------------------------------
assert os.path.exists(BEST_ADAPTER), \
    f"Adapter not found at {BEST_ADAPTER} â€” please attach the nb2-output dataset."
assert (os.path.isfile(os.path.join(BEST_ADAPTER, "adapter_config.json")) and
        (bool(_glob.glob(os.path.join(BEST_ADAPTER, "*.safetensors"))) or
         bool(_glob.glob(os.path.join(BEST_ADAPTER, "adapter_model.bin"))))), \
    f"Adapter at {BEST_ADAPTER} is incomplete â€” missing adapter_config.json or weights."

_tc_path = os.path.join(BEST_ADAPTER, "train_config.json")
assert os.path.exists(_tc_path), f"train_config.json missing from {BEST_ADAPTER}."

with open(_tc_path) as f:
    _tc = json.load(f)

assert _tc.get("schema_version") == SCHEMA_VERSION, (
    f"Adapter schema_version {_tc.get('schema_version')!r} != {SCHEMA_VERSION!r}.\\n"
    "Re-run NB2 with current data."
)
CFG.student_model_id = _tc.get("actual_model_id", CFG.student_model_id)
print(f"[PASS] Adapter schema_version verified. Base student model: {CFG.student_model_id}")

# ---- Verify NB1 manifest --------------------------------------------------
_manifest_path = os.path.join(CFG.data_dir, "data_manifest.json")
assert os.path.exists(_manifest_path), f"data_manifest.json not found in {CFG.data_dir}."
with open(_manifest_path) as f:
    _manifest = json.load(f)
assert _manifest["schema_version"] == SCHEMA_VERSION, \
    f"NB1 manifest schema {_manifest['schema_version']!r} != {SCHEMA_VERSION!r}"
print(f"[PASS] NB1 manifest schema_version verified.")
print(f"\\nðŸš€ Ready for FULL evaluation across all splits (val, test_seen, test_unseen)!\\n")


## Section 1 â€” Data Structures + Load JSONL

In [ ]:
# ============================================================================
# 1A. DATA STRUCTURES + LOADER
# ============================================================================
import json, os
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

if "SCHEMA_VERSION" not in globals():
    SCHEMA_VERSION = "v3.1"

@dataclass
class GradingRecord:
    sample_id:           str
    question_id:         str
    question_family:     str
    question_text:       str
    max_marks:           float
    rubric:              list
    student_answer:      str     # line-numbered
    student_answer_raw:  str
    mutations_applied:   list
    gold_criteria:       list
    gold_total:          float
    has_style_noise:     bool
    has_ocr_noise:       bool
    is_heavy_ocr:        bool
    needs_human_review:  bool
    execution_report:    Optional[dict] = None
    schema_version:      str = SCHEMA_VERSION
    program_id:          str = ""      # bootstrap CIs resample whole programs
    is_garbled:          bool = False  # excluded from mark accuracy, used for abstention


def load_records_jsonl(path: str) -> list:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            for k in ("sample_id", "student_answer", "student_answer_raw", "rubric", "gold_criteria"):
                if k not in d:   # no silent fallback (README rule 5)
                    raise KeyError(f"{path}: record {d.get('sample_id', '?')} has no '{k}'")
            d.setdefault("execution_report", None)
            d.setdefault("schema_version", SCHEMA_VERSION)
            d.setdefault("question_family", "")
            d.setdefault("is_heavy_ocr", False)
            d.setdefault("needs_human_review", False)
            records.append(GradingRecord(**{
                k: v for k, v in d.items()
                if k in GradingRecord.__dataclass_fields__
            }))
    return records

print(f"[PASS] GradingRecord & load_records_jsonl defined (SCHEMA_VERSION={SCHEMA_VERSION}).")


In [ ]:
# ============================================================================
# 1B. LOAD ALL SPLITS
# ============================================================================
import os, random, logging

if "CFG" not in globals():
    raise RuntimeError("CFG not found. Please run Cell 0D first.")

splits: dict = {}
# Load v2 splits with fallback to v1.1
all_target_splits = ["train", "val_seen", "val_unseen", "test_seen", "test_unseen_synth", "test_unseen_real_tasks", "val", "test_unseen"]
for sn in all_target_splits:
    fpath = os.path.join(CFG.data_dir, f"{sn}.jsonl")
    if os.path.exists(fpath):
        splits[sn] = load_records_jsonl(fpath)
        print(f"  {sn:15s}: {len(splits[sn])} records")
    else:
        print(f"  [WARN] Missing {fpath} â€” skipping {sn}")
        splits[sn] = []

if SMOKE_TEST:
    for k in splits:
        splits[k] = splits[k][:CFG.smoke_eval_samples]
    print(f"[SMOKE] Trimmed splits to â‰¤ {CFG.smoke_eval_samples} samples each.")

# Fixed random subset of 100 for zero-shot comparisons
_ZS_SEED = random.Random(CFG.seed)
ZS_SUBSETS: dict = {}
for sn, recs in splits.items():
    ZS_SUBSETS[sn] = _ZS_SEED.sample(recs, min(100, len(recs)))

print(f"[PASS] Splits loaded successfully across {len(splits)} sets.")


## Section 2 â€” Prompt + Validator (shared src/)

In [ ]:
# ============================================================================
# 2. SHARED PROMPT + VALIDATOR (Self-contained with src.prompts & src.schema)
# ============================================================================
import os, sys, json, re
from typing import Optional, List, Tuple

# Ensure /kaggle/working/cpp_grader_pkg is in sys.path
_pkg = "/kaggle/working/cpp_grader_pkg" if os.path.exists("/kaggle/working") else "./cpp_grader_pkg"
if _pkg not in sys.path:
    sys.path.insert(0, _pkg)

SCHEMA_VERSION = globals().get("SCHEMA_VERSION", "v3.1")

# ---- Canonical schema validation (standalone) ----
ABSENCE_MARKERS: Tuple[str, ...] = (
    "missing", "omit", "absent", "lacks", "not found", "fails to",
    "unhandled", "unreadable", "without", "no ", "never",
)
_LINE_PREFIX_RE = re.compile(r"^\s*\d+\|\s?")

def strip_line_number(numbered_line: str) -> str:
    return _LINE_PREFIX_RE.sub("", numbered_line, count=1)

def has_absence_marker(comment: str) -> bool:
    low = comment.lower()
    return any(m in low for m in ABSENCE_MARKERS)

def validate_grading_output(output_json: str, rubric: list, student_answer_lineno: str,
                            execution_report: Optional[dict] = None) -> Tuple[bool, List[str]]:
    errors: List[str] = []
    try:
        data = json.loads(output_json)
    except (json.JSONDecodeError, TypeError) as e:
        return False, [f"JSON parse error: {e}"]

    if not isinstance(data, dict):
        return False, ["Output is not a JSON object"]

    for key in ("criteria", "needs_human_review"):
        if key not in data:
            errors.append(f"Missing key '{key}'")
    if errors:
        return False, errors

    if not isinstance(data["criteria"], list):
        return False, ["'criteria' must be a list"]
    if not isinstance(data["needs_human_review"], bool):
        errors.append("'needs_human_review' must be a boolean")
    if "total" in data:
        errors.append("Output must not contain 'total' (sum in code)")
    if any("max" in c for c in data["criteria"] if isinstance(c, dict)):
        errors.append("Criteria must not contain 'max' (comes from rubric)")

    rubric_map = {c["id"]: c for c in rubric}
    rubric_ids = set(rubric_map)
    output_ids = set()

    answer_lines = student_answer_lineno.split("\n")
    n_lines = len(answer_lines)

    for crit in data["criteria"]:
        if not isinstance(crit, dict):
            errors.append("Criterion is not a dict")
            continue
        for k in ("id", "awarded", "comment", "evidence_lines"):
            if k not in crit:
                errors.append(f"Criterion missing key '{k}'")

        cid = crit.get("id")
        if cid not in rubric_ids:
            errors.append(f"Unknown criterion id '{cid}'")
            continue
        if cid in output_ids:
            errors.append(f"Duplicate criterion id '{cid}'")
            continue
        output_ids.add(cid)

        awarded = crit.get("awarded", -1)
        max_m = rubric_map[cid]["max_marks"]
        awarded_is_num = isinstance(awarded, (int, float)) and not isinstance(awarded, bool)
        if not awarded_is_num:
            errors.append(f"'{cid}': awarded not a number")
        else:
            if awarded < 0 or awarded > max_m:
                errors.append(f"'{cid}': awarded {awarded} not in [0, {max_m}]")
            if (awarded * 2) % 1 != 0:
                errors.append(f"'{cid}': awarded {awarded} not in 0.5 increments")

        comment = crit.get("comment", "")
        if not isinstance(comment, str) or not comment.strip():
            errors.append(f"'{cid}': comment must be non-empty")
            comment = ""
        elif len(comment.split()) < 3:
            errors.append(f"'{cid}': comment must be a substantive explanation (>= 3 words)")

        ev_lines = crit.get("evidence_lines", [])
        if not isinstance(ev_lines, list):
            errors.append(f"'{cid}': evidence_lines must be a list")
            continue
        for ln in ev_lines:
            if not isinstance(ln, int) or isinstance(ln, bool):
                errors.append(f"'{cid}': evidence_lines must contain integers, got {ln!r}")
            elif ln < 1 or ln > n_lines:
                errors.append(f"'{cid}': line {ln} out of range [1, {n_lines}]")
            elif not strip_line_number(answer_lines[ln - 1]).strip():
                errors.append(f"'{cid}': cited evidence line {ln} is empty/blank (ungrounded citation)")

        if awarded_is_num and awarded < max_m and not ev_lines and comment:
            if not has_absence_marker(comment):
                errors.append(
                    f"'{cid}': deduction made without evidence_lines, but comment "
                    "does not explain what is missing/absent"
                )

    missing = rubric_ids - output_ids
    if missing:
        errors.append(f"Missing criteria: {sorted(missing)}")

    if isinstance(execution_report, dict) and execution_report.get("compile_ok") is False:
        for c in data["criteria"]:
            if not isinstance(c, dict):
                continue
            cid = str(c.get("id", ""))
            if any(t in cid.lower() for t in ("comp", "syntax", "compile")):
                if c.get("awarded") == rubric_map.get(cid, {}).get("max_marks"):
                    errors.append(f"'{cid}': full marks awarded despite compile failure in execution report")

    return len(errors) == 0, errors

# ---- Canonical prompts ----
SYSTEM_PROMPT = (
    "You are an expert C++ Programming grader (schema " + SCHEMA_VERSION + ").\n"
    "Grade the student's answer against the rubric below.\n\n"
    "RULES\n"
    "1. Evaluate each criterion independently.\n"
    "2. Award marks only for what the student's code actually demonstrates.\n"
    "3. comment: \u226420 words explaining the assessment.\n"
    "4. evidence_lines: list of 1-based line numbers in the student code where the\n"
    "   issue appears, or [] when the criterion is fully met or the issue is\n"
    "   MISSING code (explain in comment what is absent).\n"
    "5. Output ONLY compact JSON matching the schema \u2014 no markdown, no extra text.\n\n"
    "OUTPUT SCHEMA (schema " + SCHEMA_VERSION + ")\n"
    '{"criteria":[{"id":"<id>","awarded":<number>,"comment":"<string>","evidence_lines":[<int>,...]},...],"needs_human_review":false}' + "\n\n"
    "NOTES\n"
    "- awarded must be in 0.5 increments and in [0, criterion_max].\n"
    "- total is NOT in the output; it is computed from awarded values.\n"
    "- max is NOT in the output; it comes from the rubric.\n"
)

def add_line_numbers(code: str) -> str:
    lines = code.split("\n")
    width = len(str(len(lines)))
    return "\n".join(f"{i + 1:{width}}| {line}" for i, line in enumerate(lines))

def build_exec_section(rep: Optional[dict]) -> str:
    if not isinstance(rep, dict) or not rep or rep.get("available") is False:
        return '{"available": false}'
    if not rep.get("compile_ok"):
        return json.dumps({
            "compile_ok": False,
            "compile_error": (rep.get("compile_error") or "unknown")[:400],
            "compile_lines": rep.get("compile_lines", [])[:10],
        })
    results = []
    for tc in rep.get("test_results", [])[:6]:
        results.append({
            "stdin": (tc.get("stdin") or "")[:120],
            "expected": (tc.get("expected") or "")[:120],
            "actual": (tc.get("actual") or "")[:120],
            "passed": tc.get("passed", False),
            "exit_code": tc.get("exit_code"),
            "timeout": tc.get("timeout", False),
        })
    return json.dumps({
        "compile_ok": True,
        "tests_passed": rep.get("tests_passed", 0),
        "tests_total": rep.get("tests_total", 0),
        "runtime_errors": [str(e)[:120] for e in rep.get("runtime_errors", [])[:4]],
        "test_results": results,
    })

def build_grading_messages(question_text: str, rubric: list, student_answer_lineno: str,
                           execution_report: Optional[dict]) -> list:
    rub = json.dumps(
        [{"id": c["id"], "description": c["description"], "max_marks": c["max_marks"]} for c in rubric],
        indent=2,
    )
    user_msg = (
        f"## Question\n{question_text}\n\n"
        f"## Rubric\n{rub}\n\n"
        f"## Student Answer (with line numbers)\n```\n{student_answer_lineno}\n```\n\n"
        f"## Execution Report\n{build_exec_section(execution_report)}\n\n"
        "Grade this answer. Output ONLY the compact JSON."
    )
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}]

def build_grading_prompt(rec: GradingRecord) -> list:
    return build_grading_messages(rec.question_text, rec.rubric,
                                  rec.student_answer, rec.execution_report)

print(f"[PASS] Prompt builder & validate_grading_output ready (SCHEMA_VERSION={SCHEMA_VERSION}).")


## Section 3 â€” Model Loaders

In [ ]:
# ============================================================================
# 3. MODEL LOADERS
# ============================================================================

def _bnb_cfg() -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )


def load_base_model(model_id: str, device_map="auto"):
    """Load base model in 4-bit. Raises on failure (no fallback)."""
    logger.info(f"Loading base model: {model_id}")
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    try:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=_bnb_cfg(), device_map=device_map,
            trust_remote_code=True, dtype=torch.float16, attn_implementation="sdpa",
        )
    except TypeError:
        try:
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id, quantization_config=_bnb_cfg(), device_map=device_map,
                trust_remote_code=True, torch_dtype=torch.float16, attn_implementation="sdpa",
            )
        except TypeError:
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id, quantization_config=_bnb_cfg(), device_map=device_map,
                trust_remote_code=True, torch_dtype=torch.float16,
            )
    except Exception as e:
        raise RuntimeError(
            f"Failed to load {model_id}: {e}\n"
            "No fallback model is used. Fix the error and retry."
        )
    if hasattr(mdl.config, "enable_thinking"):
        mdl.config.enable_thinking = False
    mdl.eval()
    return mdl, tok


def load_finetuned_model():
    """Load base model + LoRA adapter from BEST_ADAPTER. Raises on failure."""
    base_id = CFG.student_model_id
    logger.info(f"Loading fine-tuned model: base={base_id}  adapter={BEST_ADAPTER}")
    try:
        tok = AutoTokenizer.from_pretrained(
            BEST_ADAPTER, trust_remote_code=True, padding_side="left"
        )
    except Exception:
        tok = AutoTokenizer.from_pretrained(
            base_id, trust_remote_code=True, padding_side="left"
        )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    try:
        base = AutoModelForCausalLM.from_pretrained(
            base_id, quantization_config=_bnb_cfg(), device_map="auto",
            trust_remote_code=True, dtype=torch.float16, attn_implementation="sdpa",
        )
    except TypeError:
        try:
            base = AutoModelForCausalLM.from_pretrained(
                base_id, quantization_config=_bnb_cfg(), device_map="auto",
                trust_remote_code=True, torch_dtype=torch.float16, attn_implementation="sdpa",
            )
        except TypeError:
            base = AutoModelForCausalLM.from_pretrained(
                base_id, quantization_config=_bnb_cfg(), device_map="auto",
                trust_remote_code=True, torch_dtype=torch.float16,
            )
    except Exception as e:
        raise RuntimeError(
            f"Failed to load {base_id}: {e}\n"
            "No fallback model is used. Fix the error and retry."
        )
    if hasattr(base.config, "enable_thinking"):
        base.config.enable_thinking = False
    mdl = PeftModel.from_pretrained(base, BEST_ADAPTER)
    mdl.eval()
    return mdl, tok


## Section 4 â€” Inference Engine

In [ ]:
# ============================================================================
# 4A. BATCHED GENERATION (left-pad, batch_size=8, greedy)
# ============================================================================

def _apply_template(tokenizer, messages: list) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )


def generate_batch(model, tokenizer, texts: list, max_new_tokens: int,
                   do_sample: bool = False, temperature: float = None) -> list:
    """Left-pad batch, generate, return list of decoded strings.

    No truncation: cutting the prompt would drop the rubric/answer tail or the
    assistant header. Long prompts are logged instead."""
    _dev = getattr(model, "device", None) or next(model.parameters()).device
    inputs = tokenizer(texts, return_tensors="pt", padding=True).to(_dev)
    if inputs.input_ids.shape[1] > 4096:
        logger.warning(f"Long prompt batch: {inputs.input_ids.shape[1]} tokens")
    gen_kw  = {"max_new_tokens": max_new_tokens,
                "pad_token_id":  tokenizer.eos_token_id}
    if do_sample and temperature:
        gen_kw.update({"do_sample": True, "temperature": temperature, "top_p": 0.9})
    else:
        gen_kw.update({"do_sample": False})

    with torch.no_grad():
        out_ids = model.generate(**inputs, **gen_kw)

    prompt_len = inputs.input_ids.shape[1]
    return [
        tokenizer.decode(out_ids[i][prompt_len:], skip_special_tokens=True).strip()
        for i in range(out_ids.shape[0])
    ]


In [ ]:
# ============================================================================
# 4B. ANTI-HALLUCINATION POST-PROCESSOR (VERIFIER)
# ============================================================================
import json, re, builtins
from typing import Optional, List, Tuple

ABSENCE_MARKERS: Tuple[str, ...] = (
    "missing", "omit", "absent", "lacks", "not found", "fails to",
    "unhandled", "unreadable", "without", "no ", "never",
)
_LINE_PREFIX_RE = re.compile(r"^\s*\d+\|\s?")

def strip_line_number(numbered_line: str) -> str:
    return _LINE_PREFIX_RE.sub("", numbered_line, count=1)

def has_absence_marker(comment: str) -> bool:
    return any(m in comment.lower() for m in ABSENCE_MARKERS)

def validate_grading_output(output_json: str, rubric: list, student_answer_lineno: str,
                            execution_report: Optional[dict] = None) -> Tuple[bool, List[str]]:
    errors: List[str] = []
    try:
        data = json.loads(output_json)
    except Exception as e:
        return False, [f"JSON parse error: {e}"]
    if not isinstance(data, dict):
        return False, ["Output is not a JSON object"]
    for key in ("criteria", "needs_human_review"):
        if key not in data: errors.append(f"Missing key '{key}'")
    if errors: return False, errors
    if not isinstance(data["criteria"], list): return False, ["'criteria' must be a list"]
    if not isinstance(data["needs_human_review"], bool): errors.append("'needs_human_review' must be a boolean")
    if "total" in data: errors.append("Output must not contain 'total'")
    if any("max" in c for c in data["criteria"] if isinstance(c, dict)):
        errors.append("Criteria must not contain 'max'")

    rubric_map = {c["id"]: c for c in rubric}
    rubric_ids = set(rubric_map)
    output_ids = set()
    answer_lines = student_answer_lineno.split("\n")
    n_lines = len(answer_lines)

    for crit in data["criteria"]:
        if not isinstance(crit, dict):
            errors.append("Criterion is not a dict")
            continue
        cid = crit.get("id")
        if cid not in rubric_ids:
            errors.append(f"Unknown criterion id '{cid}'")
            continue
        if cid in output_ids:
            errors.append(f"Duplicate criterion id '{cid}'")
            continue
        output_ids.add(cid)

        awarded = crit.get("awarded", -1)
        max_m = rubric_map[cid]["max_marks"]
        if not isinstance(awarded, (int, float)) or isinstance(awarded, bool):
            errors.append(f"'{cid}': awarded not a number")
        else:
            if awarded < 0 or awarded > max_m:
                errors.append(f"'{cid}': awarded {awarded} not in [0, {max_m}]")
            if (awarded * 2) % 1 != 0:
                errors.append(f"'{cid}': awarded {awarded} not in 0.5 increments")

        comment = str(crit.get("comment", "")).strip()
        if not comment:
            errors.append(f"'{cid}': comment must be non-empty")
        elif len(comment.split()) < 3:
            errors.append(f"'{cid}': comment must be a substantive explanation (>= 3 words)")

        ev_lines = crit.get("evidence_lines", [])
        if not isinstance(ev_lines, list):
            errors.append(f"'{cid}': evidence_lines must be a list")
        else:
            for ln in ev_lines:
                if not isinstance(ln, int) or isinstance(ln, bool):
                    errors.append(f"'{cid}': evidence_lines must contain integers")
                elif ln < 1 or ln > n_lines:
                    errors.append(f"'{cid}': line {ln} out of range [1, {n_lines}]")
                elif not strip_line_number(answer_lines[ln - 1]).strip():
                    errors.append(f"'{cid}': cited evidence line {ln} is blank")

        if isinstance(awarded, (int, float)) and awarded < max_m and not ev_lines and comment:
            if not has_absence_marker(comment):
                errors.append(f"'{cid}': deduction made without evidence_lines, but comment lacks absence explanation")

    missing = rubric_ids - output_ids
    if missing: errors.append(f"Missing criteria: {sorted(missing)}")
    if isinstance(execution_report, dict) and execution_report.get("compile_ok") is False:
        for c in data["criteria"]:
            if isinstance(c, dict) and any(t in str(c.get("id","")).lower() for t in ("comp", "syntax", "compile")):
                if c.get("awarded") == rubric_map.get(str(c.get("id","")), {}).get("max_marks"):
                    errors.append(f"'{c.get('id')}': full marks awarded despite compile failure")
    return len(errors) == 0, errors

# Bind to builtins so all notebook cells have guaranteed access
builtins.validate_grading_output = validate_grading_output


def postprocess_grading(raw_json: str, rubric: list,
                        student_answer: str, execution_report: Optional[dict]) -> dict:
    """
    Deterministic verifier:
    1. Extract JSON from markdown fences or bare text.
    2. Validate schema.
    3. Clamp/snap awarded marks.
    4. Verify evidence_lines are integers within range.
    5. Sum total in code (never trust model arithmetic).
    6. Execution cross-checks â†’ needs_human_review flags.
    7. Drop hallucinated criterion IDs.
    """
    result = {
        "raw_output":          raw_json,
        "parsed":              None,
        "schema_valid_raw":    False,
        "schema_valid_final":  False,
        "errors_raw":          [],
        "errors_final":        [],
        "needs_human_review":  False,
    }

    
# Deterministic 3-step parser integration from src.parse
try:
    from src.parse import parse_and_validate_output as _parse_v2
except Exception:
    _parse_v2 = None

    # Parse with strict deterministic parser if available
    reasoning_text = ""
    feedback_text = ""
    if _parse_v2 is not None:
        p_res = _parse_v2(raw_json, rubric, student_answer)
        reasoning_text = p_res.reasoning
        feedback_text = p_res.feedback
        result["reasoning"] = reasoning_text
        result["feedback"] = feedback_text
        result["failure_types"] = p_res.failure_types
        if p_res.is_valid:
            result["schema_valid_raw"] = True
            result["schema_valid_final"] = True
            result["parsed"] = {
                "criteria": p_res.criteria,
                "total": p_res.total_marks,
                "needs_human_review": p_res.needs_human_review,
            }
            result["needs_human_review"] = p_res.needs_human_review
            return result

    # Fallback extraction
    clean = raw_json.strip()
    if "```json" in clean:
        clean = clean.split("```json")[-1].split("```")[0].strip()
    elif "```" in clean:
        parts = clean.split("```")
        clean = parts[1].strip() if len(parts) > 1 else clean
    m = re.search(r'\{.*\}', clean, re.DOTALL)
    if m:
        clean = m.group()

    # RAW parse + validate
    try:
        raw_parsed = json.loads(clean)
    except json.JSONDecodeError as e:
        result["errors_raw"]   = [f"JSON parse error: {e}"]
        result["errors_final"] = [f"JSON parse error: {e}"]
        result["needs_human_review"] = True
        return result

    result["raw_parsed"] = raw_parsed
    ok_raw, errs_raw = validate_grading_output(
        json.dumps(raw_parsed), rubric, student_answer
    )
    result["schema_valid_raw"] = ok_raw
    result["errors_raw"]       = errs_raw

    # VERIFIER repairs
    rubric_map   = {c["id"]: c for c in rubric}
    rubric_ids   = set(rubric_map)
    rubric_order = [c["id"] for c in rubric]
    n_lines      = len(student_answer.split("\n"))

    aligned  = []
    seen_ids = set()
    hr       = False

    for crit in raw_parsed.get("criteria", []):
        if not isinstance(crit, dict):
            continue
        cid = crit.get("id")

        if cid not in rubric_ids:
            result["errors_final"].append(f"Hallucinated id '{cid}' dropped.")
            hr = True
            continue
        if cid in seen_ids:
            continue
        seen_ids.add(cid)

        max_m   = float(rubric_map[cid]["max_marks"])
        awarded = crit.get("awarded", 0.0)
        if not isinstance(awarded, (int, float)):
            awarded = 0.0
            hr = True

        # Snap to 0.5 increments; DO NOT clamp -- log out-of-range (Round 3 Rule 0.1)
        awarded = round(float(awarded) * 2) / 2.0
        if awarded < 0 or awarded > max_m:
            errors.append(f"'{cid}': awarded {awarded} out of range [0, {max_m}] -- not clamped")
            result['out_of_range_count'] = result.get('out_of_range_count', 0) + 1
            hr = True
        # Keep awarded as-is (unclamped)

        # Validate evidence_lines
        ev_raw   = crit.get("evidence_lines", [])
        ev_clean = []
        if isinstance(ev_raw, list):
            for ln in ev_raw:
                if isinstance(ln, int) and 1 <= ln <= n_lines:
                    ev_clean.append(ln)
                else:
                    result["errors_final"].append(
                        f"'{cid}': evidence line {ln!r} out of range [1,{n_lines}] â€” removed."
                    )
                    hr = True
        else:
            hr = True

        comment = str(crit.get("comment", "")).strip()[:200]
        aligned.append({
            "id":             cid,
            "awarded":        awarded,
            "comment":        comment,
            "evidence_lines": ev_clean,
        })

    # Fill missing criteria
    for cid in rubric_ids:
        if cid not in seen_ids:
            result["errors_final"].append(f"Missing criterion '{cid}'; assigned 0.")
            hr = True
            aligned.append({"id": cid, "awarded": 0.0,
                            "comment": "Criterion omitted; human review required.",
                            "evidence_lines": []})

    aligned.sort(key=lambda c: rubric_order.index(c["id"])
                 if c["id"] in rubric_order else 999)

    total = round(sum(c["awarded"] for c in aligned) * 2) / 2.0

    # EXECUTION CROSS-CHECKS
    rep = execution_report or {}
    if rep.get("compile_ok") is not None:
        syn_crits = [c for c in aligned if "syntax" in c["id"] or "comp" in c["id"]]
        for sc in syn_crits:
            sc_max = float(rubric_map[sc["id"]]["max_marks"])
            if not rep.get("compile_ok") and sc["awarded"] >= sc_max:
                result["errors_final"].append(
                    f"Cross-check: compile failed but '{sc['id']}' has full marks."
                )
                hr = True

        if rep.get("tests_passed", 0) == rep.get("tests_total", 0) and rep.get("tests_total", 0) > 0:
            for c in aligned:
                if c["awarded"] == 0.0 and "syntax" not in c["id"] and "comp" not in c["id"]:
                    result["errors_final"].append(
                        f"Cross-check: all tests pass but '{c['id']}' awarded 0."
                    )
                    hr = True
                    break

        has_timeout = any("timeout" in str(e) for e in rep.get("runtime_errors", []))
        loop_crits  = [c for c in aligned if "loop" in c["id"]]
        for lc in loop_crits:
            lc_max = float(rubric_map[lc["id"]]["max_marks"])
            if has_timeout and lc["awarded"] >= lc_max:
                result["errors_final"].append(
                    f"Cross-check: timeout detected but '{lc['id']}' has full marks."
                )
                hr = True

    parsed = {
        "criteria":           aligned,
        "total":              total,
        "needs_human_review": hr,
    }
    result["parsed"]              = parsed
    result["needs_human_review"]  = hr

    ok_final, errs_final = validate_grading_output(
        json.dumps({"criteria": aligned, "needs_human_review": hr}),
        rubric, student_answer
    )
    result["schema_valid_final"] = ok_final
    result["errors_final"].extend(errs_final)
    if not ok_final:
        result["needs_human_review"] = True

    return result

In [ ]:
# ============================================================================
# 4C. INFERENCE WITH RETRY (greedy â†’ low-temp sampling)
# ============================================================================

def run_inference_one(model, tokenizer, rec: GradingRecord, cfg: InferConfig) -> dict:
    """
    Attempt greedy decoding, then up to cfg.inference_retries SAMPLING retries.
    Returns postprocessed result dict.
    """
    messages = build_grading_prompt(rec)
    text     = _apply_template(tokenizer, messages)

    for attempt in range(cfg.inference_retries + 1):
        do_sample   = (attempt > 0)
        temperature = cfg.retry_temperature if do_sample else None
        raw         = generate_batch(model, tokenizer, [text],
                                     cfg.max_new_tokens, do_sample, temperature)[0]
        result      = postprocess_grading(raw, rec.rubric,
                                          rec.student_answer, rec.execution_report)
        if result["schema_valid_final"]:
            return result
        logger.debug(f"Attempt {attempt+1} failed: {result['errors_final'][:1]}")

    result["needs_human_review"] = True
    return result

In [ ]:
# ============================================================================
# 4D. BATCH INFERENCE
# ============================================================================

def run_batch_inference(model, tokenizer, records: list, cfg: InferConfig,
                        desc: str = "Inference") -> list:
    """
    Batched generation (batch_size=8, left-pad).
    Returns one result dict per record (INVALID outputs count as wrong).
    """
    results  = []
    batch_sz = cfg.batch_size

    for batch_start in tqdm(range(0, len(records), batch_sz), desc=desc):
        batch   = records[batch_start: batch_start + batch_sz]
        texts   = [_apply_template(tokenizer, build_grading_prompt(r)) for r in batch]
        raws    = generate_batch(model, tokenizer, texts, cfg.max_new_tokens)

        for rec, raw in zip(batch, raws):
            res = postprocess_grading(raw, rec.rubric,
                                      rec.student_answer, rec.execution_report)
            if not res["schema_valid_final"]:
                # One retry with sampling
                raw2 = generate_batch(model, tokenizer, [_apply_template(tokenizer, build_grading_prompt(rec))],
                                      cfg.max_new_tokens, do_sample=True,
                                      temperature=cfg.retry_temperature)[0]
                res2 = postprocess_grading(raw2, rec.rubric,
                                           rec.student_answer, rec.execution_report)
                if res2["schema_valid_final"]:
                    res = res2

            res["sample_id"]     = rec.sample_id
            res["gold_total"]    = rec.gold_total
            res["gold_criteria"] = rec.gold_criteria
            res["max_marks"]     = rec.max_marks
            res["question_id"]   = rec.question_id
            res["mutations"]     = rec.mutations_applied
            res["is_heavy_ocr"]  = rec.is_heavy_ocr
            res["is_garbled"]    = rec.is_garbled
            res["program_id"]    = rec.program_id
            res["n_answer_lines"] = len(rec.student_answer.split("\n"))
            results.append(res)

    return results

## Section 5 â€” Evaluation Metrics

In [ ]:
# ============================================================================
# 5A. METRICS (RAW and FINAL)
# ============================================================================
from collections import defaultdict
import numpy as np
from sklearn.metrics import cohen_kappa_score, mean_absolute_error

def _graded(r: dict) -> bool:
    """Samples that count for mark accuracy. Garbled scans (README 9.3) and heavy-OCR
    samples are evaluated only for abstention."""
    return not r.get("is_garbled") and not r.get("is_heavy_ocr")


def _worst_pred(gold: float, max_marks: float) -> float:
    """Prediction with the maximum possible error (README 9.3: invalid = maximum error)."""
    return 0.0 if gold >= max_marks / 2.0 else float(max_marks)


def _final_pred(r: dict) -> float:
    if r.get("schema_valid_final") and r.get("parsed"):
        return float(r["parsed"]["total"])
    return _worst_pred(r.get("gold_total", 0.0), r.get("max_marks", 0.0))


def _raw_pred(r: dict) -> float:
    rp = r.get("raw_parsed")
    if r.get("schema_valid_raw") and isinstance(rp, dict):
        return round(sum(float(c.get("awarded", 0)) for c in rp.get("criteria", [])) * 2) / 2.0
    return _worst_pred(r.get("gold_total", 0.0), r.get("max_marks", 0.0))


def _criterion_hits(r: dict) -> tuple:
    """(correct, total) over ALL gold criteria; an invalid output gets every criterion wrong."""
    gold_map = {c["id"]: c["awarded"] for c in r.get("gold_criteria", [])}
    if not (r.get("schema_valid_final") and r.get("parsed")):
        return 0, len(gold_map)
    pred_map = {c["id"]: c["awarded"] for c in r["parsed"]["criteria"]}
    ok = sum(1 for cid, g in gold_map.items() if cid in pred_map and abs(pred_map[cid] - g) < 0.01)
    return ok, len(gold_map)


def _bootstrap_ci(results: list, stat: str, n_boot: int = 1000, seed: int = 42) -> str:
    """95% CI, resampling whole programs (README 9.5: not renderings, otherwise n is inflated)."""
    if not results:
        return "n/a"
    groups = defaultdict(list)
    for r in results:
        groups[r.get("program_id") or r["sample_id"]].append(r)
    keys = list(groups)
    num = np.zeros(len(keys)); den = np.zeros(len(keys))
    for i, k in enumerate(keys):
        for r in groups[k]:
            g, p = r["gold_total"], _final_pred(r)
            if stat == "exact":
                num[i] += abs(g - p) < 0.01; den[i] += 1
            elif stat == "mae":
                num[i] += abs(g - p); den[i] += 1
            elif stat == "criterion":
                ok, tot = _criterion_hits(r); num[i] += ok; den[i] += tot
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(keys), size=(n_boot, len(keys)))
    vals = num[idx].sum(1) / np.maximum(den[idx].sum(1), 1)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return f"[{lo:.3f}, {hi:.3f}]"


def compute_metrics(results: list, label: str = "") -> dict:
    """
    Metrics on RAW model output (before the verifier) and FINAL output (after it).
    Invalid outputs are never excluded: they get the maximum possible error.
    Garbled and heavy-OCR samples are excluded from mark accuracy and reported
    separately as abstention metrics. CIs resample whole programs.
    """
    M = {"n_total": len(results)}
    graded = [r for r in results if _graded(r)]
    M["n_graded"] = len(graded)
    M["n_programs_graded"] = len({r.get("program_id") or r["sample_id"] for r in graded})

    # â”€â”€ RAW â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    M["raw_json_validity"] = sum(1 for r in results if r.get("schema_valid_raw")) / max(len(results), 1)
    ev_total = ev_ok = 0
    for r in results:
        rp = r.get("raw_parsed")
        if not isinstance(rp, dict):
            continue
        for c in rp.get("criteria", []) if isinstance(rp.get("criteria"), list) else []:
            if not isinstance(c, dict):
                continue
            for ln in c.get("evidence_lines", []) if isinstance(c.get("evidence_lines"), list) else []:
                ev_total += 1
                ev_ok += isinstance(ln, int) and 1 <= ln <= r.get("n_answer_lines", 0)
    M["raw_hallucination_rate"] = 1.0 - ev_ok / max(ev_total, 1)
    M["raw_evidence_lines_n"] = ev_total
    raw_g = [r["gold_total"] for r in graded]
    raw_p = [_raw_pred(r) for r in graded]
    M["raw_exact_match"] = sum(1 for g, p in zip(raw_g, raw_p) if abs(g - p) < 0.01) / max(len(graded), 1)
    M["raw_mae"] = mean_absolute_error(raw_g, raw_p) if graded else float("nan")

    # â”€â”€ FINAL â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    gold = [r["gold_total"] for r in graded]
    pred = [_final_pred(r) for r in graded]
    M["final_schema_validity"] = sum(1 for r in graded if r.get("schema_valid_final")) / max(len(graded), 1)
    M["n_invalid_final"] = sum(1 for r in graded if not r.get("schema_valid_final"))
    M["final_exact_match"] = sum(1 for g, p in zip(gold, pred) if abs(g - p) < 0.01) / max(len(gold), 1)
    M["final_within_0.5"] = sum(1 for g, p in zip(gold, pred) if abs(g - p) <= 0.5) / max(len(gold), 1)
    M["final_mae"] = mean_absolute_error(gold, pred) if gold else float("nan")
    try:
        M["final_qwk"] = cohen_kappa_score([int(round(g * 2)) for g in gold],
                                           [int(round(p * 2)) for p in pred], weights="quadratic")
    except Exception:
        M["final_qwk"] = float("nan")
    ok = tot = 0
    for r in graded:
        a, b = _criterion_hits(r); ok += a; tot += b
    if tot > 0:
        M["final_per_criterion_accuracy"] = ok / tot
        # Per-criterion CI handled in guarded branch above
    else:
        M["final_per_criterion_accuracy"] = float("nan")
        M["final_per_criterion_accuracy_ci"] = "n/a"

    partial = [r for r in graded if 0 < r["gold_total"] < r.get("max_marks", 0)]
    M["n_partial"] = len(partial)
    M["partial_exact_match"] = (sum(1 for r in partial if abs(r["gold_total"] - _final_pred(r)) < 0.01)
                                / max(len(partial), 1))
    # "Correct" = full gold marks (works for synthetic and real data alike)
    correct = [r for r in graded if r["gold_total"] >= r.get("max_marks", 0) - 0.01]
    M["n_reference_or_alt_correct"] = len(correct)
    M["correct_false_penalty_rate"] = (sum(1 for r in correct if _final_pred(r) < r["gold_total"] - 0.01)
                                       / max(len(correct), 1))

    M["final_exact_match_ci"] = _bootstrap_ci(graded, "exact")
    M["final_mae_ci"] = _bootstrap_ci(graded, "mae")
    M["final_per_criterion_accuracy_ci"] = _bootstrap_ci(graded, "criterion")

    # â”€â”€ Review / abstention â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    M["final_review_flag_rate"] = sum(1 for r in graded if r.get("needs_human_review")) / max(len(graded), 1)
    nf = [r for r in graded if r.get("schema_valid_final") and not r.get("needs_human_review")]
    M["n_nonflagged"] = len(nf)
    if nf:
        M["nonflagged_exact_match"] = sum(1 for r in nf if abs(r["gold_total"] - _final_pred(r)) < 0.01) / len(nf)
        M["nonflagged_mae"] = mean_absolute_error([r["gold_total"] for r in nf], [_final_pred(r) for r in nf])
    else:
        M["nonflagged_exact_match"] = float("nan")
        M["nonflagged_mae"] = float("nan")
    garbled = [r for r in results if r.get("is_garbled")]
    heavy = [r for r in results if r.get("is_heavy_ocr") and not r.get("is_garbled")]
    M["garbled_count"] = len(garbled)
    M["garbled_flag_recall"] = sum(1 for r in garbled if r.get("needs_human_review")) / max(len(garbled), 1)
    M["heavy_ocr_count"] = len(heavy)
    M["heavy_ocr_flagged_rate"] = sum(1 for r in heavy if r.get("needs_human_review")) / max(len(heavy), 1)
    
    # ── Phase 4 Advanced Metrics ──────────────────────────────────────────────
    if gold and pred:
        # Signed error and Leniency (Pathak et al. / Diaconu et al.)
        signed_errors = [p - g for g, p in zip(gold, pred)]
        M["signed_error"] = float(np.mean(signed_errors))
        
        # Normalised by max_marks
        max_marks_list = [r.get("max_marks", 10.0) or 10.0 for r in graded]
        norm_signed = [(p - g) / max(m, 0.1) for g, p, m in zip(gold, pred, max_marks_list)]
        M["leniency"] = float(np.mean(norm_signed))
        M["mae_normalized"] = float(M["final_mae"] / max(np.mean(max_marks_list), 0.1))

        # Tolerance bands: ±3.3% and ±10% of max marks (Diaconu et al.)
        tol_3_3 = [abs(p - g) <= 0.033 * m for g, p, m in zip(gold, pred, max_marks_list)]
        tol_10 = [abs(p - g) <= 0.10 * m for g, p, m in zip(gold, pred, max_marks_list)]
        M["within_3_3_pct"] = sum(tol_3_3) / len(gold)
        M["within_10_pct"] = sum(tol_10) / len(gold)

        # Correlation metrics
        from scipy.stats import pearsonr, spearmanr, kendalltau
        try:
            if len(gold) >= 3 and len(set(gold)) > 1 and len(set(pred)) > 1:
                M["pearson_r"] = float(pearsonr(gold, pred)[0])
                M["spearman_rho"] = float(spearmanr(gold, pred)[0])
                M["kendall_tau"] = float(kendalltau(gold, pred)[0])
            else:
                M["pearson_r"] = float("nan")
                M["spearman_rho"] = float("nan")
                M["kendall_tau"] = float("nan")
        except Exception:
            M["pearson_r"] = float("nan")
            M["spearman_rho"] = float("nan")
            M["kendall_tau"] = float("nan")

        # False-penalty & false-credit rates
        full_gt = [r for r in graded if r["gold_total"] >= r.get("max_marks", 10.0) - 0.01]
        zero_gt = [r for r in graded if r["gold_total"] <= 0.5]
        M["false_penalty_rate"] = sum(1 for r in full_gt if _final_pred(r) < r["gold_total"] - 0.01) / max(len(full_gt), 1)
        M["false_credit_rate"] = sum(1 for r in zero_gt if _final_pred(r) > 1.0) / max(len(zero_gt), 1)

        # Degeneracy check: prediction variance per question
        q_preds = defaultdict(list)
        for r in graded:
            q_preds[r.get("question_id", "default")].append(_final_pred(r))
        q_vars = {qid: float(np.var(p_list)) for qid, p_list in q_preds.items() if len(p_list) > 1}
        M["min_question_pred_var"] = min(q_vars.values()) if q_vars else float("nan")
        M["is_degenerate"] = any(v < 1e-4 for v in q_vars.values()) if q_vars else False

    return M


In [ ]:
# ============================================================================
# 5B. COVERAGE VS ACCURACY TABLE
# ============================================================================

def coverage_accuracy(results: list, thresholds=(0.5, 0.7, 0.9, 1.0)) -> list:
    """
    Sort by confidence (non-flagged first, schema-valid first) and compute
    accuracy at top-k% coverage levels.
    """
    non_ocr = [r for r in results if _graded(r)]
    # Assign confidence score: 2 = valid+nonflagged, 1 = valid+flagged, 0 = invalid
    def conf(r):
        if r.get("schema_valid_final") and not r.get("needs_human_review"):
            return 2
        if r.get("schema_valid_final"):
            return 1
        return 0

    sorted_r = sorted(non_ocr, key=conf, reverse=True)
    rows = []
    for thr in thresholds:
        k    = max(1, int(len(sorted_r) * thr))
        sub  = sorted_r[:k]
        gold = [r["gold_total"] for r in sub]
        pred = [_final_pred(r) for r in sub]
        em   = sum(1 for g, p in zip(gold, pred) if abs(g - p) < 0.01) / max(len(sub), 1)
        rows.append({"coverage": thr, "n": k, "exact_match": em})
    return rows

In [ ]:
# ============================================================================
# 5C. PER-MUTATION BREAKDOWN
# ============================================================================

def per_mutation_breakdown(results: list) -> dict:
    by_mut = defaultdict(list)
    for r in results:
        if not _graded(r):
            continue
        muts = r.get("mutations", []) or ["__correct__"]
        gold = r.get("gold_total", 0.0)
        pred = _final_pred(r)
        err  = abs(gold - pred)
        for m in muts:
            by_mut[m].append(err)
    return {
        m: {"count": len(errs),
            "mae":   float(np.mean(errs)) if errs else 0.0,
            "exact": sum(1 for e in errs if e < 0.01) / max(len(errs), 1)}
        for m, errs in sorted(by_mut.items())
    }

In [ ]:
# ============================================================================
# 5D. PER-QUESTION BREAKDOWN
# ============================================================================

def per_question_breakdown(results: list) -> dict:
    by_q = defaultdict(list)
    for r in results:
        if not _graded(r):
            continue
        qid  = r.get("question_id", "?")
        gold = r.get("gold_total", 0.0)
        pred = _final_pred(r)
        by_q[qid].append(abs(gold - pred))
    return {
        q: {"count": len(errs),
            "mae":   float(np.mean(errs)) if errs else 0.0}
        for q, errs in sorted(by_q.items())
    }

In [ ]:
# ============================================================================
# 5E. PRINT HELPERS
# ============================================================================

_RAW_KEYS = ["raw_json_validity", "raw_hallucination_rate", "raw_evidence_lines_n",
             "raw_exact_match", "raw_mae"]
_METRICS_KEYS = _RAW_KEYS + [
    "n_total", "n_graded", "n_programs_graded", "n_invalid_final",
    "final_schema_validity", "final_exact_match", "final_exact_match_ci", "final_within_0.5",
    "final_mae", "final_mae_ci", "final_qwk",
    "final_per_criterion_accuracy", "final_per_criterion_accuracy_ci",
    "n_partial", "partial_exact_match",
    "n_reference_or_alt_correct", "correct_false_penalty_rate",
    "final_review_flag_rate", "n_nonflagged", "nonflagged_exact_match", "nonflagged_mae",
    "garbled_count", "garbled_flag_recall", "heavy_ocr_count", "heavy_ocr_flagged_rate",
]


def _fmt(val, col_w: int = 24) -> str:
    if isinstance(val, bool) or not isinstance(val, (int, float)):
        return f"{str(val):>{col_w}}"
    return f"{val:>{col_w}.4f}"


def print_results_table(all_metrics: dict) -> None:
    names = sorted(all_metrics)
    col_w = max(24, max(len(n) for n in names) + 3) if names else 24
    lbl_w = 35
    line = "=" * (lbl_w + col_w * len(names))
    print(f"\n{line}\nEVALUATION RESULTS (mark accuracy excludes garbled + heavy-OCR; invalid = max error)\n{line}")
    print(f"{'Metric':<{lbl_w}}" + "".join(f"{n:>{col_w}}" for n in names))
    for title, keys in [("--- RAW (before verifier) ---", _RAW_KEYS),
                        ("--- FINAL (after verifier) ---", [k for k in _METRICS_KEYS if k not in _RAW_KEYS])]:
        print(title)
        for mk in keys:
            print(f"  {mk:<{lbl_w-2}}" + "".join(_fmt(all_metrics[n].get(mk, float("nan")), col_w) for n in names))
    print(line)


def print_confusion_matrix(results: list, label: str) -> None:
    non_ocr = [r for r in results if _graded(r)]
    gold = [r["gold_total"] for r in non_ocr]
    pred = [_final_pred(r) for r in non_ocr]
    vals = sorted(set(round(v*2)/2.0 for v in gold + pred))
    vm   = {v: i for i, v in enumerate(vals)}
    gi   = [vm[round(g*2)/2.0] for g in gold]
    pi   = [vm[round(p*2)/2.0] for p in pred]
    from sklearn.metrics import confusion_matrix as _cm
    cm   = _cm(gi, pi, labels=list(range(len(vals))))
    print(f"\n-- Confusion Matrix: {label} --")
    print(f"{'':>6}", end="")
    for v in vals: print(f"{v:>6.1f}", end="")
    print()
    for i, row in enumerate(cm):
        print(f"{vals[i]:>6.1f}", end="")
        for v in row: print(f"{v:>6d}", end="")
        print()


def print_examples(results: list, label: str, n: int = 20) -> None:
    print(f"\n{'='*70}")
    print(f"EXAMPLE PREDICTIONS: {label}")
    print(f"{'='*70}")
    shown = 0
    for r in results:
        if shown >= n: break
        if not _graded(r): continue
        muts   = r.get("mutations", []) or ["correct"]
        gold_t = r.get("gold_total", "?")
        pred_t = r["parsed"]["total"] if r.get("schema_valid_final") and r.get("parsed") else "INVALID"
        print(f"\n  [{r.get('question_id','?')}] muts={muts}  "
              f"gold={gold_t}  pred={pred_t}  "
              f"review={r.get('needs_human_review',False)}")
        if r.get("parsed"):
            for pc in r["parsed"]["criteria"]:
                gm = next((c["awarded"] for c in r["gold_criteria"]
                           if c["id"] == pc["id"]), "?")
                ok = "OK" if isinstance(gm, (int, float)) and abs(pc["awarded"] - gm) < 0.01 else "XX"
                print(f"    [{ok}] {pc['id']:30s}  gold={gm}  pred={pc['awarded']}")
        else:
            print(f"    [INVALID] {r.get('errors_final',['?'])[:1]}")
        shown += 1


## Section 6 â€” Run Evaluations

In [ ]:
# ============================================================================
# 6A. SYSTEM (c): FINE-TUNED Qwen3-4B â€” main result
# ============================================================================

all_metrics: dict = {}
all_results: dict = {}

if "ft_model" not in globals() or "ft_tok" not in globals():
    logger.info("Loading fine-tuned model â€¦")
    ft_model, ft_tok = load_finetuned_model()
else:
    logger.info("Reusing loaded fine-tuned model â€¦")

for sn, recs in splits.items():
    if not recs: continue
    key = f"FT-4B_{sn}"
    logger.info(f"  {key}: {len(recs)} samples")
    res = run_batch_inference(ft_model, ft_tok, recs, CFG, desc=key)
    all_results[key]  = res
    all_metrics[key]  = compute_metrics(res, key)
    # Same 100 samples the zero-shot systems get â†’ paired comparison (README rule 2)
    _sub_ids = {r.sample_id for r in ZS_SUBSETS[sn]}
    _sub_key = f"FT-4B_{sn}@zs{len(_sub_ids)}"
    all_results[_sub_key] = [x for x in res if x["sample_id"] in _sub_ids]
    all_metrics[_sub_key] = compute_metrics(all_results[_sub_key], _sub_key)

try:
    del ft_model
except Exception:
    pass
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ============================================================================
# 6B. BASELINES (Phase 4: Retired ZS-4B / ZS-8B from synthetic loops to save compute)
# ============================================================================
logger.info("Synthetic zero-shot 4B/8B loops retired per Phase 4 protocol (evaluating zero-shot base on Quiz 3 only).")


In [ ]:
# ============================================================================
# 6C. EXECUTION-ONLY BASELINE (B_exec)
# ============================================================================
for sn in ["test_seen", "test_unseen_synth", "test_unseen_real_tasks"]:
    if sn in splits and splits[sn]:
        bx = exec_baseline(splits[sn])
        if bx:
            key = f"B_exec_{sn}"
            all_results[key] = bx
            all_metrics[key] = compute_metrics(bx, key)


## Section 7 â€” Results + Reports

In [ ]:
# ============================================================================
# 7A. RESULTS TABLE (RAW + FINAL columns)
# ============================================================================

print_results_table(all_metrics)

In [ ]:
# ============================================================================
# 7B. CONFUSION MATRICES
# ============================================================================

for key, res in all_results.items():
    print_confusion_matrix(res, key)

In [ ]:
# ============================================================================
# 7C. COVERAGE VS ACCURACY TABLE
# ============================================================================

print(f"\n{'='*70}")
print(f"{'COVERAGE VS ACCURACY (Fine-tuned 4B, val)':^70}")
print(f"{'='*70}")
_ft_val_key = next((k for k in all_results if "FT-4B_val" in k), None)
if _ft_val_key:
    _cov_rows = coverage_accuracy(all_results[_ft_val_key])
    print(f"  {'Coverage':>12}  {'N':>6}  {'ExactMatch':>12}")
    for row in _cov_rows:
        print(f"  {row['coverage']:>12.0%}  {row['n']:>6d}  {row['exact_match']:>12.3f}")

In [ ]:
# ============================================================================
# 7D. PER-MUTATION BREAKDOWN
# ============================================================================

for key in [k for k in all_results if k.startswith("FT-4B") and "@" not in k]:
    bd = per_mutation_breakdown(all_results[key])
    print(f"\n-- Per-Mutation Breakdown: {key} --")
    print(f"  {'Mutation':<35} {'N':>6} {'MAE':>8} {'Exact':>8}")
    for m, st in bd.items():
        print(f"  {m:<35} {st['count']:>6} {st['mae']:>8.3f} {st['exact']:>8.3f}")

In [ ]:
# ============================================================================
# 7E. PER-QUESTION BREAKDOWN ON test_unseen
# ============================================================================

_ft_unseen = next((k for k in all_results if "FT-4B_test_unseen" in k), None)
if _ft_unseen:
    bq = per_question_breakdown(all_results[_ft_unseen])
    print(f"\n-- Per-Question Breakdown: {_ft_unseen} --")
    print(f"  {'Question':<25} {'N':>6} {'MAE':>8}")
    for q, st in bq.items():
        print(f"  {q:<25} {st['count']:>6} {st['mae']:>8.3f}")

In [ ]:
# ============================================================================
# 7F. EXAMPLE PREDICTIONS
# ============================================================================

for key in [k for k in all_results if "FT-4B_val" in k]:
    print_examples(all_results[key], key, n=20)
    break

In [ ]:
# ============================================================================
# 7G. SAVE CSV + SUMMARY MARKDOWN
# ============================================================================

results_csv = os.path.join(CFG.output_dir, "evaluation_results.csv")
with open(results_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["system"] + _METRICS_KEYS)
    for sn, mets in sorted(all_metrics.items()):
        w.writerow([sn] + [mets.get(mk, "") for mk in _METRICS_KEYS])
logger.info(f"Results CSV â†’ {results_csv}")

# Per-sample predictions (raw output untouched + verifier result) â€” README Â§6.4
pred_dir = os.path.join(CFG.output_dir, "predictions")
os.makedirs(pred_dir, exist_ok=True)
for key, res in all_results.items():
    if "@" in key:
        continue   # subsets are filters of the full-split file
    with open(os.path.join(pred_dir, f"{key}.jsonl"), "w", encoding="utf-8") as f:
        for r in res:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
logger.info(f"Predictions â†’ {pred_dir}/")

summary_md = os.path.join(CFG.output_dir, "evaluation_summary.md")
with open(summary_md, "w", encoding="utf-8") as f:
    f.write(f"# C++ Auto-Grader Evaluation Summary\n\n")
    f.write(f"**Schema**: {SCHEMA_VERSION}  |  **Date**: {time.strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write("## Results\n\n")
    f.write("Mark accuracy excludes garbled and heavy-OCR samples; invalid outputs count as the "
            "maximum possible error. CIs: 95% bootstrap over programs. `@zs100` rows are the "
            "fine-tuned model on the same samples as the zero-shot rows (paired).\n\n")
    hdrs = ["System", "n graded", "RAW JSON%", "RAW Exact", "RAW Halluc%", "FINAL Exact (CI)",
            "FINAL MAE (CI)", "QWK", "Crit Acc (CI)", "Correct false-penalty", "Review%"]
    f.write("| " + " | ".join(hdrs) + " |\n")
    f.write("| " + " | ".join(["---"]*len(hdrs)) + " |\n")
    for sn, mets in sorted(all_metrics.items()):
        f.write(
            f"| {sn} | {mets.get('n_graded', 0)}"
            f" | {mets.get('raw_json_validity', 0):.2%}"
            f" | {mets.get('raw_exact_match', 0):.2%}"
            f" | {mets.get('raw_hallucination_rate', 0):.2%}"
            f" | {mets.get('final_exact_match', 0):.2%} {mets.get('final_exact_match_ci', '')}"
            f" | {mets.get('final_mae', 0):.3f} {mets.get('final_mae_ci', '')}"
            f" | {mets.get('final_qwk', 0):.3f}"
            f" | {mets.get('final_per_criterion_accuracy', 0):.2%} {mets.get('final_per_criterion_accuracy_ci', '')}"
            f" | {mets.get('correct_false_penalty_rate', 0):.2%} (n={mets.get('n_reference_or_alt_correct', 0)})"
            f" | {mets.get('final_review_flag_rate', 0):.2%} |\n"
        )
    f.write(f"\n## Config\n```json\n{json.dumps(_dc.asdict(CFG), indent=2, default=str)}\n```\n")

logger.info(f"Summary â†’ {summary_md}")
print(f"\n[PASS] NB3 complete.")
print(f"       CSV     â†’ {results_csv}")
print(f"       Summary â†’ {summary_md}")

## Section 8 â€” Real holdout data (professor's dataset)

Set `REAL_DATA_PATH` (env var, or edit the cell) and run sections 0â€“5, then this section.
Sections 6â€“7 (synthetic) can be skipped for a real-only run.
Freeze the adapter, prompt and code BEFORE the first real run; that run is the headline result.

In [ ]:
# ============================================================================
# 8A. REAL DATA LOADER (FAST NUCES Quiz 3 & Arbitrary Real Data)
# ============================================================================
# Supports:
# 1. Directory / dataset path containing Quiz3_1E.xlsx, question.txt, rubric.txt
#    Default Kaggle path: /kaggle/input/datasets/muhammadistafa/quiz3-1e
# 2. Direct .xlsx file (Excel with columns: Roll No, Solution, GT)
# 3. Direct .zip file (auto-extracted)
# 4. Canonical JSONL / CSV files

import os, sys, json, csv, glob as _glob, re, time, gc, torch, hashlib as _hl, shutil as _sh
import subprocess as _sp, tempfile as _tf, zipfile
import xml.etree.ElementTree as ET
from typing import Optional, List, Dict, Any

DEFAULT_KAGGLE_QUIZ3 = "/kaggle/input/datasets/muhammadistafa/quiz3-1e"

def _resolve_real_data_path() -> str:
    env_p = os.environ.get("REAL_DATA_PATH", "").strip()
    if env_p and os.path.exists(env_p):
        return env_p
    cands = [
        DEFAULT_KAGGLE_QUIZ3,
        "/kaggle/input/quiz3-1e",
        "/kaggle/input/datasets/muhammadistafa/quiz3_1e",
        "/kaggle/input/quiz3_1e",
        "/kaggle/input/quiz3",
        "quiz3_extracted",
        "Quiz3-1E",
    ]
    for c in cands:
        if os.path.exists(c):
            return c
    matches = (
        _glob.glob("/kaggle/input/**/Quiz3*.*", recursive=True) +
        _glob.glob("/kaggle/input/**/quiz3*.*", recursive=True) +
        _glob.glob("/kaggle/input/**/Quiz3*", recursive=True) +
        _glob.glob("/kaggle/input/**/quiz3*", recursive=True)
    )
    if matches:
        return sorted(matches)[0]
    return env_p or DEFAULT_KAGGLE_QUIZ3

REAL_DATA_PATH = _resolve_real_data_path()
logger.info(f"Target REAL_DATA_PATH: {REAL_DATA_PATH}")

# Canonical Question & Rubric for Quiz 3 (Denominations / Currency Change)
QUIZ3_CANONICAL_QUESTION = (
    "Write a C++ program that takes an amount in rupees (as an integer) and calculates "
    "the minimum number of currency notes needed to make that amount using the denominations: "
    "1000, 500, 100, 50, 20, 10, 5, and 1. We assume that a user will enter an amount>1000 and amount <= 9999.\n"
    "Sample input: 2896\n"
    "Sample output: Your change is below:\n"
    "1000:2\n500:1\n100:3\n50:1\n20:2\n10:0\n5:1\n1:1"
)

QUIZ3_CANONICAL_RUBRIC = [
    {"id": "q3_input", "description": "Input amount as integer", "max_marks": 1.0},
    {"id": "q3_calc", "description": "Calculate no. of notes using denominations (1000, 500, 100, 50, 20, 10, 5, 1)", "max_marks": 3.0},
    {"id": "q3_output", "description": "Display output formatted as currency change", "max_marks": 1.0},
    {"id": "q3_logic", "description": "Correct program flow/logic", "max_marks": 2.0},
]


def _read_excel_quiz(excel_path: str) -> List[Dict[str, Any]]:
    """Robust Excel reader: attempts pandas first, falls back to direct OpenXML parsing."""
    rows = []
    # 1. Try pandas
    try:
        import pandas as pd
        df = pd.read_excel(excel_path)
        for _, r in df.iterrows():
            roll = str(r.get("Roll No") or r.get("RollNo") or r.get("roll") or "").strip()
            sol = str(r.get("Solution") or r.get("solution") or r.get("Code") or "").strip()
            gt_val = r.get("GT") if pd.notna(r.get("GT")) else r.get("Marks", 0.0)
            if sol:
                try:
                    gt = float(gt_val)
                except (ValueError, TypeError):
                    gt = 0.0
                rows.append({"roll_no": roll, "student_code": sol, "teacher_total": gt})
        if rows:
            return rows
    except Exception as e:
        logger.warning(f"pandas read_excel failed ({e}); falling back to OpenXML parser.")

    # 2. Pure-Python OpenXML parser (zero external dependencies)
    with zipfile.ZipFile(excel_path, "r") as z:
        with z.open("xl/worksheets/sheet1.xml") as f:
            tree = ET.parse(f)
            root = tree.getroot()
            ns = {"main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
            sheet_data = root.find("main:sheetData", ns)
            if sheet_data is not None:
                for row_el in sheet_data.findall("main:row", ns):
                    if row_el.attrib.get("r") == "1":
                        continue
                    cells = {}
                    for c in row_el.findall("main:c", ns):
                        ref = c.attrib.get("r", "")
                        col = "".join(filter(str.isalpha, ref))
                        is_node = c.find("main:is", ns)
                        v_node = c.find("main:v", ns)
                        val = ""
                        if is_node is not None:
                            t_node = is_node.find("main:t", ns)
                            if t_node is not None and t_node.text:
                                val = t_node.text
                        elif v_node is not None and v_node.text:
                            val = v_node.text
                        cells[col] = val
                    roll = cells.get("A", "").strip()
                    sol = cells.get("B", "").strip()
                    try:
                        gt = float(cells.get("C", "0").strip())
                    except ValueError:
                        gt = 0.0
                    if sol:
                        rows.append({"roll_no": roll, "student_code": sol, "teacher_total": gt})
    return rows


def _parse_rubric_txt(text: str) -> Optional[List[Dict[str, Any]]]:
    """Parse rubric.txt if structured like '1 mark: input amount...'"""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    criteria = []
    idx = 1
    for line in lines:
        if line.lower().startswith("total"):
            continue
        m = re.match(r"^([\d\.]+)\s*marks?:\s*(.*)$", line, re.IGNORECASE)
        if m:
            marks = float(m.group(1))
            desc = m.group(2).strip()
            criteria.append({"id": f"q3_c{idx}", "description": desc, "max_marks": marks})
            idx += 1
    if criteria and sum(c["max_marks"] for c in criteria) > 0:
        return criteria
    return None


def _as_obj(v):
    return json.loads(v) if isinstance(v, str) and v.strip() else (v or None)


def _norm_rubric(rubric: list) -> list:
    out = []
    for c in rubric:
        out.append({"id": str(c["id"]),
                    "description": c.get("description", c.get("desc", "")),
                    "max_marks": float(c.get("max_marks", c.get("max")))})
    return out


def _run_test_cases_real(code: str, test_cases: list) -> dict:
    """Compile + run student's code if test_cases provided; returns {"available": False} otherwise."""
    if not test_cases:
        return {"available": False}
    if not _sh.which("g++"):
        return {"available": False}
    tmpdir = _tf.mkdtemp()
    try:
        src = os.path.join(tmpdir, "sol.cpp")
        exe = os.path.join(tmpdir, "sol")
        with open(src, "w", encoding="utf-8") as f:
            f.write(code)
        cr = _sp.run(["g++", "-std=c++17", "-O0", "-o", exe, src], capture_output=True, text=True, timeout=10)
        if cr.returncode != 0:
            lines = [(int(m.group(1)), m.group(2).strip()) for m in
                     (re.match(r"[^:]+:(\d+):\d+:\s+(.*)", l) for l in cr.stderr.splitlines()) if m]
            return {"compile_ok": False, "compile_error": cr.stderr[:1000], "compile_lines": lines[:10],
                    "tests_passed": 0, "tests_total": len(test_cases), "test_results": [], "runtime_errors": []}
        results, passed, rt_err = [], 0, []
        for tc in test_cases:
            stdin = tc.get("stdin", tc.get("in", ""))
            expected = tc.get("expected_stdout", tc.get("out", ""))
            try:
                rr = _sp.run([exe], input=stdin, capture_output=True, text=True, timeout=2)
                ok = expected.rstrip() in rr.stdout.rstrip()
                results.append({"stdin": stdin[:120], "expected": expected[:120], "actual": rr.stdout[:120],
                                "passed": ok, "exit_code": rr.returncode, "timeout": False})
                passed += ok
                if rr.returncode != 0: rt_err.append(f"exit code {rr.returncode}")
            except _sp.TimeoutExpired:
                results.append({"stdin": stdin[:120], "expected": expected[:120], "actual": "",
                                "passed": False, "exit_code": None, "timeout": True})
                rt_err.append("timeout")
        return {"compile_ok": True, "compile_error": "", "compile_lines": [], "tests_passed": passed,
                "tests_total": len(test_cases), "test_results": results, "runtime_errors": rt_err}
    finally:
        _sh.rmtree(tmpdir, ignore_errors=True)


def load_real_data(path: str) -> List[GradingRecord]:
    """Load real graded answers from Excel, directory, zip, jsonl, or csv."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Real data path not found: {path}")

    working_path = path
    if os.path.isfile(path) and path.endswith(".zip"):
        tmp_unzip = _tf.mkdtemp(prefix="quiz3_zip_")
        with zipfile.ZipFile(path, "r") as z:
            z.extractall(tmp_unzip)
        working_path = tmp_unzip

    excel_file = None
    question_file = None
    rubric_file = None

    if os.path.isdir(working_path):
        for root, _, files in os.walk(working_path):
            for fn in files:
                low = fn.lower()
                fp = os.path.join(root, fn)
                if low.endswith(".xlsx"):
                    excel_file = fp
                elif "question" in low and low.endswith(".txt"):
                    question_file = fp
                elif "rubric" in low and low.endswith(".txt"):
                    rubric_file = fp
    elif working_path.endswith(".xlsx"):
        excel_file = working_path
        parent = os.path.dirname(working_path)
        q_cand = os.path.join(parent, "question.txt")
        r_cand = os.path.join(parent, "rubric.txt")
        if os.path.isfile(q_cand): question_file = q_cand
        if os.path.isfile(r_cand): rubric_file = r_cand

    # Resolve question text
    question_text = QUIZ3_CANONICAL_QUESTION
    if question_file and os.path.isfile(question_file):
        try:
            with open(question_file, "r", encoding="utf-8") as f:
                content = f.read().strip()
                if content: question_text = content
        except Exception as e:
            logger.warning(f"Failed to read question file {question_file}: {e}")

    # Resolve rubric
    rubric = QUIZ3_CANONICAL_RUBRIC
    if rubric_file and os.path.isfile(rubric_file):
        try:
            with open(rubric_file, "r", encoding="utf-8") as f:
                rtxt = f.read().strip()
                parsed_rub = _parse_rubric_txt(rtxt)
                if parsed_rub: rubric = parsed_rub
        except Exception as e:
            logger.warning(f"Failed to read rubric file {rubric_file}: {e}")

    max_marks = sum(c["max_marks"] for c in rubric)
    records = []
    seen = set()

    # Case A: Excel file (.xlsx)
    if excel_file and os.path.isfile(excel_file):
        logger.info(f"Loading real data from Excel: {excel_file}")
        raw_rows = _read_excel_quiz(excel_file)
        for i, row in enumerate(raw_rows):
            code = row["student_code"]
            roll = row["roll_no"] or f"student_{i+1}"
            gt_total = round(float(row["teacher_total"]) * 2) / 2.0
            sid = f"real_q3_{roll}"
            if sid in seen:
                sid = f"{sid}_{i}"
            seen.add(sid)

            records.append(GradingRecord(
                sample_id=sid,
                question_id="quiz3_denominations",
                question_family="real",
                question_text=question_text,
                max_marks=max_marks,
                rubric=rubric,
                student_answer=add_line_numbers(code),
                student_answer_raw=code,
                mutations_applied=[],
                gold_criteria=[],
                gold_total=gt_total,
                has_style_noise=False,
                has_ocr_noise=False,
                is_heavy_ocr=False,
                needs_human_review=False,
                execution_report={"available": False},
                program_id=f"student:{_hl.sha256(str(roll).encode()).hexdigest()[:12]}",
            ))

    # Case B: Standard JSONL or CSV file
    elif working_path.endswith(".jsonl") or working_path.endswith(".csv"):
        if working_path.endswith(".jsonl"):
            with open(working_path, encoding="utf-8") as f:
                rows = [json.loads(l) for l in f if l.strip()]
        else:
            with open(working_path, newline="", encoding="utf-8") as f:
                rows = list(csv.DictReader(f))
        for i, row in enumerate(rows):
            rub = _norm_rubric(_as_obj(row.get("rubric", row.get("rubric_json")))) or rubric
            code = row.get("student_code", row.get("student_answer"))
            q_txt = row.get("question_text", row.get("question")) or question_text
            total = row.get("human_total", row.get("teacher_total", row.get("GT")))
            sid = row.get("sample_id") or f"real_{i}"
            roll = row.get("student_id", row.get("Roll No", sid))
            gt_total = round(float(total) * 2) / 2.0
            tcs = _as_obj(row.get("test_cases", row.get("test_cases_json"))) or []
            records.append(GradingRecord(
                sample_id=sid,
                question_id=str(row.get("question_id", "real")),
                question_family="real",
                question_text=q_txt,
                max_marks=sum(c["max_marks"] for c in rub),
                rubric=rub,
                student_answer=add_line_numbers(code),
                student_answer_raw=code,
                mutations_applied=[],
                gold_criteria=[],
                gold_total=gt_total,
                has_style_noise=False,
                has_ocr_noise=False,
                is_heavy_ocr=False,
                needs_human_review=False,
                execution_report=_run_test_cases_real(code, tcs),
                program_id=f"student:{_hl.sha256(str(roll).encode()).hexdigest()[:12]}",
            ))
    else:
        raise ValueError(f"Could not find Quiz3_1E.xlsx, .jsonl, or .csv in: {path}")

    logger.info(f"Loaded {len(records)} real student records (Max Marks = {max_marks}).")
    return records


def exec_baseline(records: list) -> list:
    """B_exec baseline: returns [] if no test cases are available."""
    out = []
    for rec in records:
        rep = rec.execution_report or {}
        if rep.get("available") is False:
            continue
        ratio = 0.0 if not rep.get("compile_ok") else rep.get("tests_passed", 0) / max(rep.get("tests_total", 1), 1)
        crit = [{"id": c["id"], "awarded": round(c["max_marks"] * ratio * 2) / 2.0,
                 "comment": "execution-only baseline", "evidence_lines": []} for c in rec.rubric]
        out.append({"sample_id": rec.sample_id, "raw_output": "", "raw_parsed": None,
                    "schema_valid_raw": False, "schema_valid_final": True, "needs_human_review": False,
                    "parsed": {"criteria": crit, "total": round(sum(c["awarded"] for c in crit) * 2) / 2.0},
                    "gold_total": rec.gold_total, "gold_criteria": rec.gold_criteria,
                    "max_marks": rec.max_marks, "question_id": rec.question_id, "mutations": [],
                    "is_heavy_ocr": False, "is_garbled": False, "program_id": rec.program_id,
                    "n_answer_lines": len(rec.student_answer.split("\n"))})
    return out

In [ ]:

# ============================================================================
# 8B. REAL EVALUATION (FAST NUCES Quiz 3 — Dev & Test Splits, Raw vs Clean)
# ============================================================================

real_data_target = _resolve_real_data_path()

if os.path.exists(real_data_target):
    logger.info(f"Executing Real Holdout Evaluation on: {real_data_target}")
    real_records_all = load_real_data(real_data_target)
    real_dir = os.path.join(CFG.output_dir, "real")
    os.makedirs(os.path.join(real_dir, "predictions"), exist_ok=True)
    real_results, real_metrics = {}, {}

    # Load frozen split if available
    split_file = "quiz3_split.json"
    if not os.path.isfile(split_file):
        _split_cands = _glob.glob("/kaggle/input/**/quiz3_split.json", recursive=True)
        if _split_cands:
            split_file = _split_cands[0]
    quiz3_dev_ids, quiz3_test_ids = set(), set()
    if os.path.isfile(split_file):
        with open(split_file, "r") as f:
            q3_split = json.load(f)
            quiz3_dev_ids = {s.get("hash") or s.get("roll_no") or s.get("rollNo") for s in q3_split.get("dev", [])}
            quiz3_test_ids = {s.get("hash") or s.get("roll_no") or s.get("rollNo") for s in q3_split.get("test", [])}
        logger.info(f"Loaded frozen Quiz 3 split: {len(quiz3_dev_ids)} dev, {len(quiz3_test_ids)} test students.")

    def _in_split(rec, id_set):
        sid = str(rec.sample_id)
        clean_roll = sid.replace("real_q3_", "")
        h = _hl.sha256(clean_roll.encode()).hexdigest()
        return (sid in id_set) or (clean_roll in id_set) or (h in id_set)

    dev_records = [r for r in real_records_all if _in_split(r, quiz3_dev_ids)]
    test_records = [r for r in real_records_all if _in_split(r, quiz3_test_ids)]
    if not dev_records or not test_records:
        dev_records, test_records = real_records_all[:23], real_records_all[23:]

    # 1. Fine-Tuned Model (Our Core Result) on All 46, Dev, and Test
    mdl, tok = load_finetuned_model()
    for name, recs in [("FT-4B_quiz3_all46", real_records_all), ("FT-4B_quiz3_dev", dev_records), ("FT-4B_quiz3_test", test_records)]:
        res = run_batch_inference(mdl, tok, recs, CFG, desc=name)
        real_results[name] = res
        real_metrics[name] = compute_metrics(res, name)

    # 2. Zero-Shot Base Model on Quiz 3 (Cheap single base run per Phase 4)
    logger.info("Evaluating Zero-Shot Base Model on Quiz 3 dev/test...")
    base_mdl, base_tok = load_base_model(CFG.student_model_id)
    for name, recs in [("ZS_Base_quiz3_all46", real_records_all), ("ZS_Base_quiz3_dev", dev_records), ("ZS_Base_quiz3_test", test_records)]:
        res = run_batch_inference(base_mdl, base_tok, recs, CFG, desc=name)
        real_results[name] = res
        real_metrics[name] = compute_metrics(res, name)
    del base_mdl; gc.collect(); torch.cuda.empty_cache()

    # 3. Execution Baseline
    _bx = exec_baseline(real_records_all)
    if _bx:
        real_results["B_exec_quiz3"] = _bx
        real_metrics["B_exec_quiz3"] = compute_metrics(_bx, "B_exec_quiz3")

    # 4. Error Analysis Dump on Dev (Disagreements >= 10% of max marks)
    dev_res = real_results.get("FT-4B_quiz3_dev", [])
    dev_dump = []
    for r in dev_res:
        gold = r.get("gold_total", 0.0)
        pred = _final_pred(r)
        diff = abs(pred - gold)
        if diff >= 0.10 * r.get("max_marks", 7.0):
            dev_dump.append({
                "student_id": r.get("sample_id"),
                "instructor_marks": gold,
                "predicted_marks": pred,
                "diff": diff,
                "needs_human_review": r.get("needs_human_review"),
                "reasoning_excerpt": r.get("reasoning", "")[:300],
                "student_code_preview": r.get("student_answer_raw", "")[:200]
            })

    dump_path = os.path.join(real_dir, "dev_error_analysis.json")
    with open(dump_path, "w", encoding="utf-8") as f:
        json.dump(dev_dump, f, indent=2)
    logger.info(f"Dev error analysis dump ({len(dev_dump)} disagreements >= 10% max) saved to {dump_path}")

    # Display results
    print_results_table(real_metrics)

    # Save outputs
    for key, res in real_results.items():
        with open(os.path.join(real_dir, "predictions", f"{key}.jsonl"), "w", encoding="utf-8") as f:
            for r in res:
                f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")

    with open(os.path.join(real_dir, "real_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(real_metrics, f, indent=2, default=str)
